# 25 · The static filters, scored as the told models are

Notebook 1 measured the static filters on an earlier patch set, with PIL, in RGB PSNR. That set is gone, and its
numbers cannot be read beside the models': bicubic is 34.98 dB there and 33.55 dB of luma on notebook 21's patches. This
notebook scores the filters where notebooks 22–24 score the told models: **notebook 21's 9,628 validation patches,
through `training.evaluate`** (`helpers/metrics.py`: SSIM-Y, luma, chroma, wPSNR and RGB PSNR), with the halo cut off
the same way. Bicubic here is `models.bicubic`, the skip every model is built on, and it has to score what notebook 24
printed for it.

It also asks whether **bicubic + unsharp mask**, notebook 1's best filter, should be the bar instead of bicubic. The
mask is scored at notebook 1's setting and at the setting that does best on the training split.

1. **The filters**: torch's, PIL's (in float, no 8-bit rounding), notebook 1's pipeline exactly, and an unsharp mask on
   the models' bicubic.
2. **Tuning the unsharp mask** on 8,000 training patches.
3. **Scores** on the validation patches, overall and by encoder and frame type.

Needs: notebook 21 (`qp_slice_pairs/`). Trains nothing and exports nothing. **Limits:** notebook 21's targets are
video only, with no desktop screens or photos, which are where text and hard edges are.

In [1]:
"""Settings and data."""

import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, sources, training
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_slice_pairs"    # notebook 21
NOTEBOOK_24_BICUBIC = {"ssim": 0.9045, "y": 33.55}         # what notebook 24 printed for bicubic on these patches
SWEEP_PATCHES = 8000                                       # training patches the unsharp mask is tuned on
RADII, AMOUNTS = (0.5, 1.0, 1.5), (0.1, 0.2, 0.4, 0.8)     # the blur's standard deviation (HR px), the mask's strength
BATCH_SIZE = 128
SEED = 0

training.seed_everything(SEED)
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE, ENCODERS = manifest["scale"], manifest["encoders"]
trim = training.trimmer(manifest)
val_set = patches.PatchPairs(PAIRS_DIR, "val", manifest, augment=False)
train_set = patches.PatchPairs(PAIRS_DIR, "train", manifest, augment=False)
labels = {name: np.load(PAIRS_DIR / manifest["splits"]["val"][name]) for name in ("encoder", "intra")}
print(f"train {len(train_set)} patches, val {len(val_set)} patches")

device  cuda NVIDIA GeForce RTX 3090
scale   2
patch   72 px LR in (64 px + a halo of 4) -> 128 px HR out, 1 frames each
train 34364 patches, val 9628 patches


## 1. The filters

Every filter is a module from a batch of LR patches to its ×2 picture, so `training.evaluate` scores it exactly as it
scores a model: the output clamped to [0, 1], the halo trimmed, and the scores averaged per patch.

- **torch** (`F.interpolate`): nearest, bilinear and bicubic. Bicubic is `models.bicubic` (a = −0.75), the skip in every
  model.
- **PIL** (`Image.resize`) on each channel as a float image, so nothing is rounded to 8 bits: Hamming, Lanczos-3, and
  PIL's own bicubic (a = −0.5, a softer kernel than torch's). Box is left out because at ×2 it is nearest.
- **Notebook 1's pipeline exactly**: 8-bit PIL bicubic, then PIL's `UnsharpMask(radius=1, percent=80, threshold=0)`.
- **An unsharp mask on the models' bicubic**: the picture plus `amount` times its difference from a Gaussian blur of
  standard deviation `radius`, which is how PIL defines its radius. The threshold is 0 and the edge is repeated. It is
  written in torch so it can be tuned; at notebook 1's setting it is checked against PIL's below.

In [2]:
"""The filters."""


class Pil(nn.Module):
    """A PIL resampling filter on each colour channel as a float ("F") image, so no 8-bit rounding."""

    def __init__(self, resample):
        super().__init__()
        self.resample = resample

    def forward(self, x):
        n, c, h, w = x.shape
        out = np.empty((n, c, h * SCALE, w * SCALE), np.float32)
        for i, image in enumerate(x.cpu().numpy()):
            for j, channel in enumerate(image):
                out[i, j] = np.asarray(Image.fromarray(channel).resize((w * SCALE, h * SCALE), self.resample))
        return torch.from_numpy(out).to(x.device)


class Notebook1Unsharp(nn.Module):
    """Notebook 1's best filter as notebook 1 ran it: 8-bit PIL bicubic, then PIL's unsharp mask."""

    def forward(self, x):
        out = []
        for image in (x.cpu().numpy() * 255).round().astype(np.uint8).transpose(0, 2, 3, 1):
            picture = Image.fromarray(image).resize((image.shape[1] * SCALE, image.shape[0] * SCALE), Image.BICUBIC)
            out.append(np.asarray(picture.filter(ImageFilter.UnsharpMask(radius=1.0, percent=80, threshold=0))))
        return torch.from_numpy(np.stack(out).transpose(0, 3, 1, 2).astype(np.float32) / 255).to(x.device)


def gaussian(sigma):
    """A normalised 1-D Gaussian out to 3 standard deviations."""
    reach = int(np.ceil(3 * sigma))
    taps = torch.arange(-reach, reach + 1, dtype=torch.float32)
    kernel = torch.exp(-taps ** 2 / (2 * sigma ** 2))
    return kernel / kernel.sum()


def unsharp(picture, radius, amount):
    """`picture` + `amount` (`picture` - its Gaussian blur), per channel, the edge repeated."""
    kernel = gaussian(radius).to(picture.device)
    reach, channels = len(kernel) // 2, picture.shape[1]
    blurred = F.pad(picture, (reach, reach, reach, reach), mode="replicate")
    blurred = F.conv2d(blurred, kernel.view(1, 1, 1, -1).expand(channels, 1, 1, -1), groups=channels)
    blurred = F.conv2d(blurred, kernel.view(1, 1, -1, 1).expand(channels, 1, -1, 1), groups=channels)
    return picture + amount * (picture - blurred)


class Unsharp(nn.Module):
    """`models.bicubic`, then `unsharp`."""

    def __init__(self, radius, amount):
        super().__init__()
        self.radius, self.amount = radius, amount

    def forward(self, x):
        return unsharp(models.bicubic(x, SCALE), self.radius, self.amount)


# the torch mask against PIL's, on the same 8-bit bicubic pictures (inside the halo, where both read real pixels)
apart = []
for index in range(0, len(val_set), len(val_set) // 50):
    lr, _ = val_set[index]
    picture = (models.bicubic(lr[None], SCALE).clamp(0, 1)[0].permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    theirs = np.asarray(Image.fromarray(picture).filter(ImageFilter.UnsharpMask(radius=1.0, percent=80, threshold=0)))
    ours = unsharp(torch.from_numpy(picture).permute(2, 0, 1)[None].float(), 1.0, 0.8).clamp(0, 255)
    apart.append(np.abs(theirs - ours[0].permute(1, 2, 0).numpy())[8:-8, 8:-8])
apart = np.stack(apart)
print(f"torch's unsharp mask against PIL's (radius 1, 80%): {apart.mean():.2f} levels apart on average, "
      f"{apart.max():.1f} at most, on {len(apart)} patches")

FILTERS = {
    "nearest": models.Resize(SCALE, "nearest"),
    "bilinear": models.Resize(SCALE, "bilinear"),
    "bicubic": models.Resize(SCALE, "bicubic"),
    "Hamming (PIL)": Pil(Image.HAMMING),
    "Lanczos-3 (PIL)": Pil(Image.LANCZOS),
    "bicubic (PIL, a = -0.5)": Pil(Image.BICUBIC),
    "notebook 1: PIL bicubic + unsharp 1 px, 80%": Notebook1Unsharp(),
    "bicubic + unsharp 1 px, 80%": Unsharp(1.0, 0.8),
}

torch's unsharp mask against PIL's (radius 1, 80%): 0.29 levels apart on average, 2.8 at most, on 51 patches


## 2. Tuning the unsharp mask

Notebook 1's 80% at radius 1 was never chosen; it was a default. Here every radius and amount in the grid is scored on
8,000 training patches (a fixed random draw) beside plain bicubic. The setting with the best luma PSNR is kept, since
luma PSNR is the score notebook 1's mask won on. The best by SSIM-Y, the score the models are selected by, is printed
too.

In [3]:
"""The grid on the training split."""

pick = np.random.default_rng(SEED).choice(len(train_set), SWEEP_PATCHES, replace=False)
sweep_loader = DataLoader(Subset(train_set, pick), batch_size=BATCH_SIZE)

sweep = {(0.0, 0.0): training.evaluate(models.Resize(SCALE, "bicubic").to(DEVICE), sweep_loader, trim)}
for radius in RADII:
    for amount in AMOUNTS:
        sweep[radius, amount] = training.evaluate(Unsharp(radius, amount).to(DEVICE), sweep_loader, trim)

base = sweep[0.0, 0.0]
print(f"{SWEEP_PATCHES} training patches: SSIM-Y / luma dB, and over bicubic")
print(f"  {'bicubic':<26}{base['ssim']:.4f} / {base['y']:.2f}")
for (radius, amount), scores in sweep.items():
    if radius:
        print(f"  {f'unsharp {radius:g} px, {amount:.0%}':<26}{scores['ssim']:.4f} / {scores['y']:.2f}"
              f"    {scores['ssim'] - base['ssim']:+.4f} / {scores['y'] - base['y']:+.2f}")

by_psnr = max(sweep, key=lambda setting: sweep[setting]["y"])
by_ssim = max(sweep, key=lambda setting: sweep[setting]["ssim"])
describe = lambda setting: "plain bicubic" if not setting[0] else f"unsharp {setting[0]:g} px, {setting[1]:.0%}"
print(f"\nbest by luma PSNR: {describe(by_psnr)}; best by SSIM-Y: {describe(by_ssim)}")
TUNED = by_psnr
FILTERS[f"bicubic + unsharp {TUNED[0]:g} px, {TUNED[1]:.0%} (tuned)"] = Unsharp(*TUNED)

8000 training patches: SSIM-Y / luma dB, and over bicubic
  bicubic                   0.9024 / 33.14
  unsharp 0.5 px, 10%       0.9023 / 33.15    -0.0001 / +0.01
  unsharp 0.5 px, 20%       0.9022 / 33.16    -0.0002 / +0.01
  unsharp 0.5 px, 40%       0.9019 / 33.16    -0.0005 / +0.02
  unsharp 0.5 px, 80%       0.9013 / 33.17    -0.0011 / +0.03
  unsharp 1 px, 10%         0.9020 / 33.17    -0.0004 / +0.03
  unsharp 1 px, 20%         0.9015 / 33.18    -0.0009 / +0.04
  unsharp 1 px, 40%         0.9001 / 33.16    -0.0023 / +0.02
  unsharp 1 px, 80%         0.8960 / 32.94    -0.0064 / -0.21
  unsharp 1.5 px, 10%       0.9018 / 33.18    -0.0006 / +0.04
  unsharp 1.5 px, 20%       0.9008 / 33.17    -0.0016 / +0.03
  unsharp 1.5 px, 40%       0.8979 / 33.02    -0.0045 / -0.13
  unsharp 1.5 px, 80%       0.8891 / 32.33    -0.0133 / -0.82

best by luma PSNR: unsharp 1.5 px, 10%; best by SSIM-Y: plain bicubic


## 3. Scores on the validation patches

Every filter on the 9,628 validation patches, absolute and against bicubic. Bicubic has to read what notebook 24 printed
on the same patches; it is checked before anything else is printed.

In [4]:
"""Every filter on the validation split."""

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
scores = {name: training.evaluate(module.to(DEVICE), val_loader, trim) for name, module in FILTERS.items()}

base = scores["bicubic"]
assert abs(base["ssim"] - NOTEBOOK_24_BICUBIC["ssim"]) < 5e-5 and abs(base["y"] - NOTEBOOK_24_BICUBIC["y"]) < 5e-3, \
    f"bicubic scores {base['ssim']:.4f} / {base['y']:.2f}, not notebook 24's"
print(f"bicubic matches notebook 24: {base['ssim']:.4f} SSIM-Y, {base['y']:.2f} dB luma\n")

print(f"{'':<52}{'ssim-y':>8}{'psnr-y':>8}{'psnr-u':>8}{'psnr-v':>8}{'wpsnr':>8}{'rgb':>8}   over bicubic")
for name, s in scores.items():
    print(f"{name:<52}{s['ssim']:8.4f}{s['y']:8.2f}{s['u']:8.2f}{s['v']:8.2f}{s['wpsnr']:8.2f}{s['rgb']:8.2f}"
          f"   {s['ssim'] - base['ssim']:+.4f} / {s['y'] - base['y']:+.2f} dB")

bicubic matches notebook 24: 0.9045 SSIM-Y, 33.55 dB luma

                                                      ssim-y  psnr-y  psnr-u  psnr-v   wpsnr     rgb   over bicubic
nearest                                               0.8883   32.01   43.76   43.69   32.98   30.83   -0.0162 / -1.54 dB
bilinear                                              0.9016   32.94   44.04   43.97   33.87   31.60   -0.0030 / -0.61 dB
bicubic                                               0.9045   33.55   43.99   43.93   34.46   32.08   +0.0000 / +0.00 dB
Hamming (PIL)                                         0.8963   32.58   43.89   43.82   33.53   31.31   -0.0082 / -0.97 dB
Lanczos-3 (PIL)                                       0.9047   33.69   43.99   43.93   34.59   32.18   +0.0002 / +0.14 dB
bicubic (PIL, a = -0.5)                               0.9048   33.50   44.01   43.95   34.41   32.04   +0.0003 / -0.05 dB
notebook 1: PIL bicubic + unsharp 1 px, 80%           0.9044   33.66   43.87   43.80   34.56 

In [5]:
"""By the encoder and frame type a patch came from: SSIM-Y / luma dB, and the same over bicubic."""

GROUPS = [("all", np.ones(len(val_set), bool))] + [
    (f"{encoder} {kind}", (labels["encoder"] == e) & (labels["intra"] == (kind == "I")))
    for e, encoder in enumerate(ENCODERS) for kind in "PI"]
SHOWN = ["bicubic", "Lanczos-3 (PIL)", "bicubic + unsharp 1 px, 80%", next(n for n in FILTERS if n.endswith("(tuned)"))]

by_group = {}
for group, keep in GROUPS:
    loader = DataLoader(Subset(val_set, np.flatnonzero(keep)), batch_size=BATCH_SIZE)
    by_group[group] = {name: training.evaluate(FILTERS[name].to(DEVICE), loader, trim) for name in SHOWN}

print(f"{'':<12}{'patches':>8}" + "".join(f"{name[:30]:>34}" for name in SHOWN))
for (group, keep) in GROUPS:
    row = by_group[group]
    cells = [f"{row[n]['ssim']:.4f} / {row[n]['y']:.2f}" + (
        "" if n == "bicubic" else f" ({row[n]['ssim'] - row['bicubic']['ssim']:+.4f})") for n in SHOWN]
    print(f"{group:<12}{keep.sum():8}" + "".join(f"{cell:>34}" for cell in cells))

             patches                           bicubic                   Lanczos-3 (PIL)       bicubic + unsharp 1 px, 80%    bicubic + unsharp 1.5 px, 10% 
all             9628                    0.9045 / 33.55          0.9047 / 33.69 (+0.0002)          0.8991 / 33.36 (-0.0055)          0.9042 / 33.60 (-0.0004)
x264 P          4192                    0.8933 / 32.93          0.8929 / 33.04 (-0.0004)          0.8829 / 32.55 (-0.0104)          0.8919 / 32.93 (-0.0014)
x264 I           700                    0.8932 / 32.93          0.8926 / 33.02 (-0.0006)          0.8833 / 32.73 (-0.0099)          0.8918 / 32.96 (-0.0013)
nvenc P         4032                    0.9164 / 34.19          0.9173 / 34.37 (+0.0009)          0.9159 / 34.17 (-0.0005)          0.9170 / 34.28 (+0.0006)
nvenc I          704                    0.9149 / 34.19          0.9156 / 34.34 (+0.0007)          0.9143 / 34.23 (-0.0006)          0.9155 / 34.29 (+0.0006)


## 4. What it says

Run 2026-11-15 on notebook 21's 9,628 validation patches (video through x264 and NVENC). The told 24×2's row is notebook
24's, on the same patches with the same code:

| | SSIM-Y | luma dB | over bicubic |
|---|---|---|---|
| bicubic (`models.bicubic`, the models' skip) | 0.9045 | 33.55 | |
| Lanczos-3 | 0.9047 | 33.69 | +0.0002 / +0.14 |
| notebook 1's pipeline: PIL bicubic + unsharp 1 px, 80% | 0.9044 | 33.66 | −0.0002 / +0.10 |
| bicubic + unsharp 1.5 px, 10% (tuned on the training split) | 0.9042 | 33.60 | −0.0004 / +0.05 |
| bicubic + unsharp 1 px, 80% | 0.8991 | 33.36 | −0.0055 / −0.19 |
| bilinear | 0.9016 | 32.94 | −0.0030 / −0.61 |
| told 24×2 (notebooks 22–24) | 0.9289 | 35.07 | +0.0243 / +1.52 |

1. **Bicubic scores what notebook 24 printed** (0.9045 SSIM-Y, 33.55 dB luma), so these numbers and the told models'
   can be read together. Notebook 1's 34.98 dB was RGB PSNR, on a different patch set.
2. **No unsharp mask beats bicubic on SSIM-Y.** On the training split every setting in the grid scores under plain
   bicubic, and the harder it sharpens, the further under it goes. On luma PSNR the best setting (1.5 px, 10%) gains
   +0.05 dB on validation. Notebook 1's 80% at 1 px, applied to the models' bicubic, loses 0.0055 SSIM-Y and 0.19 dB.
3. **Notebook 1's win came partly from PIL's softer bicubic.** PIL's kernel (a = −0.5) is 0.05 dB of luma under torch's
   (a = −0.75). Notebook 1's pipeline is +0.16 dB over PIL's own bicubic but only +0.10 over the models', and it loses
   SSIM-Y to both.
4. **Sharpening costs most where there is no loop filter.** On x264's frames, which the host sends unfiltered, the 80%
   mask loses 0.0104 SSIM-Y (P frames) and 0.0099 (I frames). On NVENC's deblocked frames it loses 0.0005–0.0006,
   and the tuned mask gains 0.0006. That is what sharpening a stream's blocks would look like.
5. **The best static filter is Lanczos-3**: +0.14 dB of luma and +0.0002 SSIM-Y over bicubic. Neither ONNX `Resize` nor
   a GPU sampler has it. Measured against it, the told 24×2's lead would be +1.38 dB and +0.0242 instead of +1.52 and
   +0.0243.

**Bicubic stays the bar.** It is the skip every model is built on, the GPU computes it for nothing, and no static
filter is more than 0.14 dB above it. A sharpened bicubic is worse on SSIM-Y, the score the models are chosen by.

**Not measured:** desktop screens and photos. Notebook 21's targets are video only, and text and hard edges might
reward sharpening differently.